# CrackSense — What Happens Inside SegFormer

**Standalone explanation notebook.** No training, no dataset loading.
Just loads the trained `best.pt` checkpoint and shows exactly what happens
to one image at every stage — from raw pixels to binary crack mask.

Includes a detailed explanation of how Otsu thresholding works.

## 1. Load the Trained Model

In [ ]:
import os, sys, io
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation
from PIL import Image

# ---- Config (must match what the model was trained with) ----
MODEL_NAME = "nvidia/mit-b0"
IMG_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "checkpoints/best.pt"

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_properties(0).name}")

# ---- Load model architecture ----
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "background", 1: "crack"},
    label2id={"background": 0, "crack": 1},
    ignore_mismatched_sizes=True,
)

# ---- Load trained weights ----
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

# ---- Read checkpoint info ----
saved_config = ckpt.get("config", {})
best_dice = ckpt.get("best_dice", "?")
best_epoch = ckpt.get("epoch", "?")
calibrated_threshold = float(ckpt.get("best_threshold", 0.50))

print(f"\nCheckpoint loaded: {CHECKPOINT_PATH}")
print(f"  Trained model:    {saved_config.get('MODEL_NAME', MODEL_NAME)}")
print(f"  Best epoch:       {best_epoch}")
print(f"  Best val Dice:    {best_dice}")
print(f"  Calibrated threshold: {calibrated_threshold}")

# ---- Helper from training notebook ----
def binary_logits_from_two_class(logits):
    """For a 2-class model, crack logit - background logit = binary crack logit."""
    return logits[:, 1] - logits[:, 0]

# ---- Inference preprocessing (same as training notebook) ----
infer_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print("\nReady. Upload an image below to see the full pipeline.")


## 2. Load an Image

In [ ]:
# --- Set your image path here ---
IMAGE_PATH = "WhatsApp Image 2026-08-17 at 22.19.55.jpeg"  # <-- change this to your image

# Load it
pil_img = Image.open(IMAGE_PATH).convert("RGB")
np_img = np.array(pil_img)
orig_h, orig_w = np_img.shape[:2]

print(f"Loaded: {IMAGE_PATH}")
print(f"Size: {orig_h} × {orig_w} × 3")

plt.figure(figsize=(6, 6))
plt.imshow(np_img)
plt.title(f"Original Image — {orig_h}×{orig_w}", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()


## 3. Preprocessing — What the Model Actually Receives

The model doesn't see the raw image. It sees a resized, normalized tensor.
Here's each step.

In [ ]:
# Step 1: Resize
resized = A.Resize(IMG_SIZE, IMG_SIZE)(image=np_img)["image"]

# Step 2: Normalize (ImageNet mean/std — because the encoder was pretrained on ImageNet)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
normalized_float = (resized.astype(np.float32) / 255.0 - mean) / std

# Step 3: To tensor
augmented = infer_tf(image=np_img)
tensor_img = augmented["image"]  # shape: (3, 256, 256)
input_tensor = tensor_img.unsqueeze(0).to(DEVICE)  # shape: (1, 3, 256, 256)

# For display: undo normalization
disp_norm = np.clip(tensor_img.numpy().transpose(1, 2, 0) * std + mean, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

axes[0].imshow(np_img)
axes[0].set_title(f"Step 1: Original\n{orig_h}×{orig_w}×3", fontsize=11, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(resized)
axes[1].set_title(f"Step 2: Resize to {IMG_SIZE}×{IMG_SIZE}\n(all images same size for the model)", fontsize=10, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(disp_norm)
axes[2].set_title("Step 3: ImageNet Normalization\nmean=[.485,.456,.406] std=[.229,.224,.225]", fontsize=9, fontweight="bold")
axes[2].axis("off")

axes[3].imshow(tensor_img.numpy()[0], cmap="viridis")
axes[3].set_title(f"Step 4: Final tensor (channel 0)\nshape: {tuple(tensor_img.shape)}", fontsize=9, fontweight="bold")
axes[3].axis("off")

plt.suptitle("Preprocessing: Raw Image → Model Input", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Original:    {orig_h} × {orig_w} × 3  (uint8, 0-255)")
print(f"Resized:     {IMG_SIZE} × {IMG_SIZE} × 3  (uint8, 0-255)")
print(f"Normalized:  {IMG_SIZE} × {IMG_SIZE} × 3  (float32, shifted to ImageNet distribution)")
print(f"Final tensor: {tuple(input_tensor.shape)}  (batch × channels × height × width)")
print(f"\nWhy ImageNet normalization?")
print(f"  The MiT-B2 encoder was pretrained on ImageNet with these exact mean/std values.")
print(f"  If we don't match them, the pretrained weights receive out-of-distribution inputs")
print(f"  and perform badly — like speaking English to someone trained only in Japanese.")


## 4. Inside the Encoder — 4 Hierarchical Stages

The MiT-B2 encoder processes the image through 4 stages.
Each stage produces a feature map at a different resolution.
Early stages see fine details (edges), later stages see broad context (semantics).

In [ ]:
# Extract intermediate feature maps from every encoder stage
with torch.no_grad():
    with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
        encoder_out = model.segformer.encoder(
            input_tensor,
            output_hidden_states=True,
            return_dict=True,
        )
        hidden_states = encoder_out.hidden_states  # tuple of 4 tensors

stage_info = [
    ("Stage 1 — Edges & fine texture",
     "Captures low-level features: edges, gradients, the fine linear texture\n"
     "of a crack against smooth concrete. Highest spatial resolution.\n"
     "Think of it as: 'what does each small patch look like?'"),

    ("Stage 2 — Local crack segments",
     "Slightly larger receptive field. Detects short connected crack segments,\n"
     "small structural features, local patterns.\n"
     "Think of it as: 'do nearby patches form a line or a blob?'"),

    ("Stage 3 — Junctions & larger patterns",
     "Captures crack junctions, branching points, larger contextual patterns.\n"
     "Can distinguish a crack from a similar-looking mortar line or shadow.\n"
     "Think of it as: 'what is this region doing?'"),

    ("Stage 4 — Global semantics",
     "The deepest, most abstract representation. Understands 'this whole area\n"
     "reads as cracked wall' vs 'this is just dirt/shadow/texture.'\n"
     "Think of it as: 'what IS this part of the image?'"),
]

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))

# Show input
axes[0].imshow(disp_norm)
axes[0].set_title("Input to encoder", fontsize=10, fontweight="bold")
axes[0].axis("off")

dim_table = []
for idx, (hs, (title, _)) in enumerate(zip(hidden_states, stage_info)):
    feat = hs[0]  # (C, H, W)
    # Aggregate channels by L2 norm — gives a 2D "activation intensity" map
    activation = feat.norm(dim=0).cpu().numpy()
    C, H, W = feat.shape
    dim_table.append((idx + 1, H, W, C))

    axes[idx + 1].imshow(activation, cmap="inferno")
    axes[idx + 1].set_title(f"{title}\n{C} ch × {H}×{W}", fontsize=9, fontweight="bold")
    axes[idx + 1].axis("off")

plt.suptitle("MiT-B2 Encoder — What Each Stage Sees", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print detailed descriptions
for idx, ((title, desc), (_, H, W, C)) in enumerate(zip(stage_info, dim_table)):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"  Tensor shape: 1 × {C} channels × {H} × {W}")
    print(f"{'='*60}")
    print(f"  {desc}")

# Resolution table
print(f"\n\n  The key pattern — spatial resolution shrinks, semantic depth grows:")
print(f"  ┌───────────┬────────┬───────┬──────────┐")
print(f"  │   Stage   │ Height │ Width │ Channels │")
print(f"  ├───────────┼────────┼───────┼──────────┤")
for s, H, W, C in dim_table:
    print(f"  │  Stage {s}  │   {H:>3} │  {W:>3} │     {C:>3}  │")
print(f"  └───────────┴────────┴───────┴──────────┘")
print(f"\n  Stage 1: lots of spatial detail, shallow understanding")
print(f"  Stage 4: little spatial detail, deep semantic understanding")
print(f"  The decoder's job is to combine BOTH — so the final mask")
print(f"  is both spatially precise AND semantically correct.")


## 5. Decoder → Probability Map

The MLP decoder takes all 4 stage outputs, fuses them, and produces
a **continuous probability** for each pixel: how likely is it to be a crack?

In [ ]:
# Full forward pass to get logits
with torch.no_grad():
    with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
        outputs = model(pixel_values=input_tensor, output_hidden_states=True)
        logits = outputs.logits  # shape: (1, 2, H/4, W/4)

# Upsample to full input size
logits_full = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE),
                            mode="bilinear", align_corners=False)

# Convert 2-class logits to single binary crack score
binary_logit = binary_logits_from_two_class(logits_full.float())  # (1, H, W)

# Sigmoid turns logit into probability (0 to 1)
prob_map = torch.sigmoid(binary_logit)[0].cpu().numpy()  # (H, W)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Decoder output (logit — can be negative or positive)
logit_np = binary_logit[0].cpu().numpy()
axes[0].imshow(logit_np, cmap="RdBu_r")
axes[0].set_title("Decoder output: raw logit\n"
                  "Red = model thinks crack\n"
                  "Blue = model thinks background", fontsize=10, fontweight="bold")
axes[0].axis("off")

# Probability map (after sigmoid — always 0 to 1)
axes[1].imshow(prob_map, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("After sigmoid: probability map\n"
                  "Black = 0% crack chance\n"
                  "White = 100% crack chance", fontsize=10, fontweight="bold")
axes[1].axis("off")

# Overlay
overlay = disp_norm.copy()
red_overlay = np.zeros_like(overlay)
red_overlay[:, :, 0] = prob_map
axes[2].imshow(np.clip(overlay * 0.55 + red_overlay * 0.45, 0, 1))
axes[2].set_title("Probability overlaid on image\n(brighter red = higher crack probability)", fontsize=10, fontweight="bold")
axes[2].axis("off")

plt.suptitle("Decoder → Sigmoid → Crack Probability Map", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"  Raw decoder output shape: {tuple(logits.shape)}")
print(f"  After upsampling:         {tuple(logits_full.shape)}")
print(f"  Binary logit range:       [{logit_np.min():.2f}, {logit_np.max():.2f}]")
print(f"  After sigmoid (prob):     [{prob_map.min():.4f}, {prob_map.max():.4f}]")
print(f"  Mean probability:         {prob_map.mean():.4f}")
print(f"\n  IMPORTANT: This is NOT a binary mask yet.")
print(f"  Every pixel has a probability between 0 and 1.")
print(f"  We still need to decide: above what probability do we call it 'crack'?")
print(f"  That decision is the THRESHOLD — explained next.")


## 6. How Otsu Thresholding Works — Step by Step

This is the key section. We explain exactly how Otsu decides the threshold,
with the actual math and the actual numbers from our probability map.

**The core question:** we have a probability map where each pixel is a number
between 0 and 1. We need to draw a line: above = crack, below = background.
Where should that line be?

**Two approaches:**
- **Calibrated threshold:** sweep many thresholds on the validation set, pick the one
  that maximizes Dice score globally. Same value for every image.
- **Otsu threshold:** for each individual image, look at its probability distribution
  and automatically find the best split point. Different for every image.

In [ ]:
# ============================================================
# OTSU THRESHOLDING — EXPLAINED WITH REAL NUMBERS
# ============================================================

print("=" * 65)
print("HOW OTSU WORKS — on our actual probability map")
print("=" * 65)

# Step 1: Look at the distribution
print("""
STEP 1: Look at the probability distribution

  Every pixel has a crack probability between 0 and 1.
  Most pixels are background (probability near 0).
  A few pixels are crack (probability near 1).
  The histogram shows this distribution.
""")

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
counts, bin_edges, patches = ax.hist(prob_map.ravel(), bins=256, range=(0, 1),
                                      color="#1E2761", edgecolor="none", alpha=0.85)
ax.set_xlabel("Crack Probability", fontsize=12)
ax.set_ylabel("Number of Pixels", fontsize=12)
ax.set_title("Probability Distribution of Our Image", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

total_pixels = prob_map.size
bg_pixels = (prob_map < 0.5).sum()
cr_pixels = (prob_map >= 0.5).sum()
print(f"  Total pixels: {total_pixels:,}")
print(f"  Pixels with prob < 0.5: {bg_pixels:,} ({100*bg_pixels/total_pixels:.1f}%)")
print(f"  Pixels with prob ≥ 0.5: {cr_pixels:,} ({100*cr_pixels/total_pixels:.1f}%)")

# Step 2: Otsu's algorithm
print("""
STEP 2: What Otsu does

  Otsu tries EVERY possible threshold T from 0 to 1.
  For each T, it splits the pixels into two groups:
    Group A: pixels with probability ≤ T  (background)
    Group B: pixels with probability > T  (crack)

  For each split, it asks:
    "How different are the average probabilities of group A vs group B?"

  This is measured by the "between-class variance":
    σ² = wA × wB × (μA − μB)²

  where:
    wA = fraction of pixels in group A
    wB = fraction of pixels in group B
    μA = mean probability of group A
    μB = mean probability of group B

  Otsu picks the T that MAXIMIZES this variance.
  In other words: the split that makes the two groups
  as different from each other as possible.
""")

# Step 3: Actually compute Otsu and show the math
prob_u8 = (prob_map * 255).astype(np.uint8)
otsu_val_u8, otsu_binary_cv = cv2.threshold(prob_u8, 0, 255,
                                             cv2.THRESH_BINARY + cv2.THRESH_OTSU)
otsu_threshold = otsu_val_u8 / 255.0

# Manually compute the between-class variance at several thresholds to show the curve
thresholds_to_try = np.arange(0.01, 1.0, 0.005)
variances = []
flat = prob_map.ravel()
for t in thresholds_to_try:
    group_a = flat[flat <= t]
    group_b = flat[flat > t]
    if len(group_a) == 0 or len(group_b) == 0:
        variances.append(0)
        continue
    wA = len(group_a) / len(flat)
    wB = len(group_b) / len(flat)
    muA = group_a.mean()
    muB = group_b.mean()
    var = wA * wB * (muA - muB) ** 2
    variances.append(var)

variances = np.array(variances)
best_idx = np.argmax(variances)
manual_otsu = thresholds_to_try[best_idx]

print(f"STEP 3: The result")
print(f"  Otsu threshold (OpenCV):  {otsu_threshold:.4f}")
print(f"  Otsu threshold (manual):  {manual_otsu:.4f}")
print(f"  (These should be very close — both use the same algorithm)")
print()

# Show the variance curve
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: the variance curve — Otsu picks the peak
axes[0].plot(thresholds_to_try, variances, color="#1E2761", linewidth=2)
axes[0].axvline(manual_otsu, color="#E63946", linewidth=2.5, linestyle="--",
                label=f"Otsu T = {manual_otsu:.3f}")
axes[0].set_xlabel("Threshold T", fontsize=12)
axes[0].set_ylabel("Between-class variance σ²", fontsize=12)
axes[0].set_title("Otsu picks the T that maximizes this curve\n"
                  "(= the split that separates the two groups most)",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=11)
axes[0].set_xlim(0, 1)

# Right: histogram with both thresholds
axes[1].hist(prob_map.ravel(), bins=256, range=(0, 1),
             color="#1E2761", edgecolor="none", alpha=0.85)
axes[1].axvline(otsu_threshold, color="#E63946", linewidth=2.5, linestyle="--",
                label=f"Otsu T = {otsu_threshold:.3f}")
axes[1].axvline(calibrated_threshold, color="#2A9D8F", linewidth=2.5, linestyle="-.",
                label=f"Calibrated T = {calibrated_threshold:.3f}")
axes[1].set_xlabel("Crack Probability", fontsize=12)
axes[1].set_ylabel("Number of Pixels", fontsize=12)
axes[1].set_title("Both thresholds on the histogram", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].set_xlim(0, 1)

plt.suptitle("How Otsu Finds the Threshold", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Show what happens at the Otsu threshold
group_a = flat[flat <= otsu_threshold]
group_b = flat[flat > otsu_threshold]
print(f"\n  At Otsu T = {otsu_threshold:.3f}:")
print(f"    Background group: {len(group_a):,} pixels, mean prob = {group_a.mean():.4f}")
print(f"    Crack group:      {len(group_b):,} pixels, mean prob = {group_b.mean():.4f}")
print(f"    Separation:       |{group_a.mean():.4f} − {group_b.mean():.4f}| = {abs(group_a.mean()-group_b.mean()):.4f}")

print("""
  KEY POINT — Otsu does NOT use any fixed position.
  It does NOT assume the threshold should be 0.5 or 0.8 or any specific value.
  It looks at THIS specific image's probability distribution and finds
  the split point where background and crack pixels are maximally separated.

  Different images → different distributions → different Otsu thresholds.

  This is why it's called "adaptive" — it adapts to each image individually,
  unlike the calibrated threshold which is the same for every image.
""")


## 7. Final Result — Binary Crack Mask

Now we apply both thresholds to the probability map and compare.

In [ ]:
# Apply both thresholds
calib_mask = (prob_map > calibrated_threshold).astype(np.uint8)
otsu_mask  = (prob_map > otsu_threshold).astype(np.uint8)

# Resize to original image size for overlay
calib_full = cv2.resize(calib_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
otsu_full  = cv2.resize(otsu_mask,  (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

# Overlay calibrated mask on original
overlay_calib = np_img.copy()
overlay_calib[calib_full == 1] = [230, 57, 70]
blended_calib = cv2.addWeighted(np_img, 0.6, overlay_calib, 0.4, 0)

# Overlay otsu mask on original
overlay_otsu = np_img.copy()
overlay_otsu[otsu_full == 1] = [42, 157, 143]
blended_otsu = cv2.addWeighted(np_img, 0.6, overlay_otsu, 0.4, 0)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: probability → calibrated mask → overlay
axes[0, 0].imshow(prob_map, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("Probability Map\n(continuous, 0–1)", fontsize=11, fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(calib_mask, cmap="gray", vmin=0, vmax=1)
axes[0, 1].set_title(f"Calibrated Mask\nT = {calibrated_threshold:.3f} (same for all images)", fontsize=11, fontweight="bold")
axes[0, 1].axis("off")

axes[0, 2].imshow(blended_calib)
axes[0, 2].set_title("Calibrated: overlay on original", fontsize=11, fontweight="bold")
axes[0, 2].axis("off")

# Row 2: original → otsu mask → overlay
axes[1, 0].imshow(np_img)
axes[1, 0].set_title("Original Image", fontsize=11, fontweight="bold")
axes[1, 0].axis("off")

axes[1, 1].imshow(otsu_mask, cmap="gray", vmin=0, vmax=1)
axes[1, 1].set_title(f"Otsu Mask\nT = {otsu_threshold:.3f} (adaptive per image)", fontsize=11, fontweight="bold")
axes[1, 1].axis("off")

axes[1, 2].imshow(blended_otsu)
axes[1, 2].set_title("Otsu: overlay on original", fontsize=11, fontweight="bold")
axes[1, 2].axis("off")

plt.suptitle("Final Comparison — Calibrated vs Otsu Thresholding", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

calib_pct = 100.0 * calib_mask.sum() / calib_mask.size
otsu_pct  = 100.0 * otsu_mask.sum() / otsu_mask.size

print(f"  Calibrated threshold: {calibrated_threshold:.3f} → {calib_pct:.2f}% crack pixels")
print(f"  Otsu threshold:       {otsu_threshold:.3f} → {otsu_pct:.2f}% crack pixels")
print()
if otsu_threshold < calibrated_threshold:
    print(f"  Otsu picked a LOWER threshold → more pixels classified as crack → more sensitive")
else:
    print(f"  Otsu picked a HIGHER threshold → fewer pixels classified as crack → more conservative")
print()
print(f"  NOTE: This comparison is VISUAL ONLY.")
print(f"  The calibrated threshold was optimized on the full validation set")
print(f"  using Dice score — it is the one used for final reporting.")
print(f"  Otsu is an experimental per-image alternative for comparison.")
